# Xarray: sea surface temperatures

This lab explores climate data in the NetCDF format with xarray: the sea surface temperature over the whole globe, month by month, from 1970 to 2021. You will read the file, select a point and a region by their coordinates, compute the climatology and the anomalies that reveal El Niño, then write a result to NetCDF and to Zarr, chunk by chunk, before looking at the HDF5 that hides under the NetCDF.

The exercises build on each other: each one uses results of the previous ones, which their solutions compute. The last one, an animation, is optional.

## Installation

Colab doesn't have Zarr: the next cell installs it. xarray, Dask, h5py and h5netcdf are already there.

In [ ]:
!pip install -q zarr

## Imports

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

## The data

NOAA's *Extended Reconstructed Sea Surface Temperature* (ERSST) dataset, version 5, reconstructs the sea surface temperature every month since 1854, on a 2° × 2° grid, from the measurements of ships and buoys ([overview](https://www.ncei.noaa.gov/products/extended-reconstructed-sst)). The `xarray.tutorial` module provides an excerpt of it, from 1970 to 2021: a 19 MB NetCDF file, downloaded on the first call, then kept in a cache. A value is the mean temperature of one month in one cell of the grid, in degrees Celsius.

`open_dataset` returns a `Dataset`; its `sst` variable (*sea surface temperature*), a `DataArray`, is the one of the whole lab.

In [ ]:
ds = xr.tutorial.open_dataset("ersstv5")
sst = ds.sst
ds

## 1. Explore the NetCDF file

The display of the `Dataset`, above, sums up the file. Answer the following questions, writing the code that shows each answer:

- what are the dimensions of `sst`, and how many values does it hold?
- what period does it cover, and what is the grid's step, in degrees?
- in what unit is the temperature (the attributes of `sst`)?
- what share of the cells of a map has no value, and why?
- how does the file store `sst` on disk: is it compressed, and in chunks of what shape (its `encoding`)? In what unit are the dates written in the file?

Finally, plot the map of the first month with `.plot()`.

**Useful documentation**

- [`DataArray.isnull`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.isnull.html)
- [Reading encoded data](https://docs.xarray.dev/en/stable/user-guide/io.html#reading-encoded-data)

In [ ]:
# Your code here

### Additional hints

`sst.sizes` gives the size of each dimension, `sst.size` the number of values. `ds.time.values[[0, -1]]` gives the first and the last dates; the difference of two neighbouring latitudes, the step. `sst.attrs` is a dictionary, as are `sst.encoding` (keys `zlib` and `chunksizes`) and `ds.time.encoding` (key `units`). `.isnull()` returns an array of booleans, whose mean is the share of missing values: a little under a third.

```python
print(sst.sizes, sst.size)
print(ds.time.values[[0, -1]])
print(float(ds.lat[0] - ds.lat[1]), float(...))
print(sst.attrs[...])
first_map = sst.isel(time=0)
print(float(first_map.isnull().mean()))
print(sst.encoding[...], sst.encoding[...], ds.time.encoding[...])
first_map.plot()
```

### Solution

In [ ]:
print(sst.sizes, sst.size)
print(ds.time.values[[0, -1]])
print(float(ds.lat[0] - ds.lat[1]), float(ds.lon[1] - ds.lon[0]))
print(sst.attrs["units"], sst.attrs["long_name"])
first_map = sst.isel(time=0)
print(float(first_map.isnull().mean()))
print(sst.encoding["zlib"], sst.encoding["chunksizes"], ds.time.encoding["units"])
first_map.plot()
# 624 months × 89 latitudes × 180 longitudes: 9,996,480 values.
# From January 1970 to December 2021, on a 2° grid in latitude and in longitude.
# In degrees Celsius (degC): monthly means.
# 31% of the cells have no value: the land, and the sea ice, with no sea to measure.
# Compressed (zlib), in chunks of one map, (1, 89, 180): reading a month reads one chunk.
# The dates are written in days since 1 January 1800: xarray decodes them.

## 2. Select by label

With `sel`, extract:

- the series of temperatures at Brest (48.4° N, 4.5° W), in the nearest cell. Plot it, then compute the mean temperature of each month of the year (`groupby("time.month")`): which month is the coldest, which the warmest, and how far apart are they?
- the Niño 3.4 region of the equatorial Pacific, from 5° S to 5° N and from 170° W to 120° W, then its mean temperature of each month, over all its cells: the `nino34` series, which the next exercises use.

Beware: the file's longitudes go from 0 to 360 eastward, and its latitudes are stored from north to south.

**Useful documentation**

- [`DataArray.sel`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html)
- [`DataArray.idxmax`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.idxmax.html)

In [ ]:
# Your code here

### Additional hints

A western longitude $x$ is written $360 - x$: 4.5° W becomes 355.5, and 170° W becomes 190. `sel(lat=..., lon=..., method="nearest")` picks the nearest cell. A slice `slice(start, stop)` follows the coordinate's order: on decreasing latitudes, `slice(-5, 5)` returns an empty array, with no error. `idxmin()` and `idxmax()` give the label (here, the month) of the minimum and of the maximum. `mean(["lat", "lon"])` averages over both dimensions at once. At Brest, the coldest and the warmest months are 7 to 8 °C apart.

```python
brest = sst.sel(lat=..., lon=..., method="nearest")
brest.plot()
brest_months = brest.groupby(...).mean()
print(brest_months.idxmin().item(), brest_months.idxmax().item(), ...)
nino34 = sst.sel(lat=slice(..., ...), lon=slice(..., ...)).mean(...)
print(nino34.sizes)
```

### Solution

In [ ]:
brest = sst.sel(lat=48.4, lon=360 - 4.5, method="nearest")
print(float(brest.lat), float(brest.lon))
brest.plot()
brest_months = brest.groupby("time.month").mean()
print(brest_months.round(1).values)
gap = float(brest_months.max() - brest_months.min())
print(brest_months.idxmin().item(), brest_months.idxmax().item(), round(gap, 1))
nino34 = sst.sel(lat=slice(5, -5), lon=slice(190, 240)).mean(["lat", "lon"])
print(nino34.sizes)
# The cell nearest to Brest is centred on 48° N, 356° E (4° W).
# The coldest month is March (10.2 °C), the warmest August (18.1 °C): 7.9 °C apart.
# nino34 has 624 values, one per month.

## 3. Climatology and anomalies

The temperature of the Niño 3.4 region follows a yearly cycle: to compare one year with another, you first have to remove that cycle. Compute:

- the *climatology* of `nino34`: its mean of each month of the year (12 values), over the 1991-2020 reference period;
- the *anomalies* `nino_anomalies`: the gap between each month of `nino34` and the climatology of its month (624 values);
- the five months with the strongest anomalies: which years are they from?
- NOAA's ONI (*Oceanic Niño Index*), the moving average of the anomalies over three months, centred. NOAA declares an El Niño episode when it exceeds +0.5 °C five months in a row, La Niña when it falls below −0.5 °C. Plot the anomalies and the ONI on the same chart, with both thresholds: what were the three strongest El Niño episodes?

**Useful documentation**

- [`DataArray.sortby`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sortby.html)
- [`DataArray.rolling`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.rolling.html)

In [ ]:
# Your code here

### Additional hints

The climatology: `sel(time=slice("1991", "2020"))`, then `groupby("time.month").mean()`. The subtraction `nino34.groupby("time.month") - climatology` takes off each month the climatology of its month. `nino_anomalies.sortby(nino_anomalies, ascending=False)` sorts the months from the strongest anomaly to the weakest, `isel(time=slice(0, 5))` keeps the first five, and `.time.dt.strftime("%Y-%m")` shows their dates readably. `rolling(time=3, center=True).mean()` computes the ONI; `oni.groupby("time.year").max()`, its maximum of each year, which you sort as the anomalies. An episode often spans two years. `plt.axhline(0.5)` draws a threshold. The strongest anomaly exceeds +2.5 °C.

```python
climatology = nino34.sel(time=...).groupby(...).mean()
nino_anomalies = nino34.groupby(...) - climatology
strongest = nino_anomalies.sortby(...).isel(time=slice(0, 5))
print(strongest.time.dt.strftime("%Y-%m").values, strongest.round(2).values)
oni = nino_anomalies.rolling(...).mean()
max_per_year = oni.groupby("time.year").max()
print(max_per_year.sortby(...).isel(year=slice(0, 6)).values)
nino_anomalies.plot(color="lightgray", label="Monthly anomaly")
oni.plot(label="ONI")
plt.axhline(...)
plt.axhline(...)
plt.legend()
```

### Solution

In [ ]:
climatology = nino34.sel(time=slice("1991", "2020")).groupby("time.month").mean()
nino_anomalies = nino34.groupby("time.month") - climatology
print(climatology.round(2).values)
strongest = nino_anomalies.sortby(nino_anomalies, ascending=False).isel(
  time=slice(0, 5)
)
print(strongest.time.dt.strftime("%Y-%m").values, strongest.round(2).values)
oni = nino_anomalies.rolling(time=3, center=True).mean()
max_per_year = oni.groupby("time.year").max()
top = max_per_year.sortby(max_per_year, ascending=False).isel(year=slice(0, 6))
print(top.year.values, top.round(2).values)
nino_anomalies.plot(color="lightgray", label="Monthly anomaly")
oni.plot(label="ONI (3-month mean)")
plt.axhline(0.5, color="red", linestyle="--")
plt.axhline(-0.5, color="blue", linestyle="--")
plt.ylabel("Niño 3.4 anomaly (°C)")
plt.legend()
# The climatology peaks in May (27.9 °C) and falls to 26.6 °C in December-January.
# The five strongest anomalies: November 2015 to January 2016 (+2.7 °C at most),
# then November 1997 and January 1998.
# The three strongest El Niño episodes: 2015-2016 (ONI of +2.64 °C), 1997-1998
# (+2.35 °C) and 1982-1983 (+2.15 °C).

## 4. Compute by chunks, write to NetCDF and to Zarr

Open the file again with `xr.tutorial.open_dataset("ersstv5", chunks={"time": 120})`: each variable becomes a Dask array, in chunks of 120 months, and the computations become lazy. On the whole map this time, compute the anomalies of `sst`: the 1991-2020 climatology of each cell, then the gap between each month and the climatology of its month. Display `.data` before and after this computation: nothing is computed yet, but what became of the chunks?

Group the chunks by 120 months (`chunk`), name the result `anomaly` and give it the `long_name` and `units` attributes that describe it, then write it to `anomalies.nc` (`to_netcdf`) and to `anomalies.zarr` (`to_zarr`): writing is what triggers the computation, chunk by chunk.

Compare the size of the two, and count the files of the Zarr directory. Read them back (`xr.open_dataset`, `xr.open_zarr`): do they hold the same values? Finally, from the Zarr, plot the map of the anomalies of December 2015.

**Useful documentation**

- [`DataArray.chunk`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.chunk.html)
- [`DataArray.to_netcdf`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.to_netcdf.html)
- [`DataArray.to_zarr`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.to_zarr.html)
- [`xarray.open_zarr`](https://docs.xarray.dev/en/stable/generated/xarray.open_zarr.html)
- [`pathlib.Path.rglob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)

In [ ]:
# Your code here

### Additional hints

Computing the anomalies is exercise 3's computation, on the whole `sst` instead of `nino34`. The `groupby` subtraction cuts the array into chunks of one month: 624 chunks, which would make 624 Zarr files. It also adds a `month` coordinate, which `drop_vars("month")` removes. `.chunk({"time": 120})` groups the chunks, `.rename("anomaly")` names the array, and `anomalies.attrs = {...}` replaces the attributes inherited from `sst`, which would describe a temperature. `to_zarr(..., mode="w")` replaces an existing directory, if you run the cell again; `consolidated=False`, when writing as when reading, avoids a warning: the Zarr 3 format has no consolidated metadata. `Path(path).stat().st_size` gives the size of a file, in bytes; that of a directory is the sum of those of the files `rglob("*")` finds in it (`is_file()`). `identical` compares the values, the coordinates and the attributes.

```python
ds_chunks = xr.tutorial.open_dataset("ersstv5", chunks=...)
sst_chunks = ds_chunks.sst
print(sst_chunks.data)
map_climatology = sst_chunks.sel(...).groupby(...).mean()
anomalies = (sst_chunks.groupby(...) - map_climatology).drop_vars("month")
print(anomalies.data)
anomalies = anomalies.chunk(...).rename(...)
anomalies.attrs = {"long_name": ..., "units": ...}
anomalies.to_netcdf(...)
anomalies.to_zarr(..., mode="w", consolidated=False)
nc_size = ...
zarr_files = [f for f in Path(...).rglob("*") if f.is_file()]
zarr_size = ...
print(nc_size, zarr_size, len(zarr_files))
nc_back = xr.open_dataset(...)
zarr_back = xr.open_zarr(..., consolidated=False)
print(nc_back.anomaly.identical(...))
zarr_back.anomaly.sel(...).plot(cmap="RdBu_r")
```

### Solution

In [ ]:
ds_chunks = xr.tutorial.open_dataset("ersstv5", chunks={"time": 120})
sst_chunks = ds_chunks.sst
print(sst_chunks.data)
reference = sst_chunks.sel(time=slice("1991", "2020"))
map_climatology = reference.groupby("time.month").mean()
anomalies = (sst_chunks.groupby("time.month") - map_climatology).drop_vars("month")
print(anomalies.data)
anomalies = anomalies.chunk({"time": 120}).rename("anomaly")
anomalies.attrs = {
  "long_name": "Sea surface temperature anomaly (1991-2020 baseline)",
  "units": "degC",
}
anomalies.to_netcdf("anomalies.nc")
anomalies.to_zarr("anomalies.zarr", mode="w", consolidated=False)
nc_size = Path("anomalies.nc").stat().st_size
zarr_files = [f for f in Path("anomalies.zarr").rglob("*") if f.is_file()]
zarr_size = sum(f.stat().st_size for f in zarr_files)
print(f"NetCDF: {nc_size / 1e6:.1f} MB")
print(f"Zarr: {zarr_size / 1e6:.1f} MB in {len(zarr_files)} files")
nc_back = xr.open_dataset("anomalies.nc")
zarr_back = xr.open_zarr("anomalies.zarr", consolidated=False)
print(zarr_back.anomaly.data)
print(nc_back.anomaly.identical(zarr_back.anomaly))
zarr_back.anomaly.sel(time="2015-12").plot(cmap="RdBu_r")
# After the subtraction, the chunks are one month long: (1, 89, 180), 624 chunks.
# The NetCDF weighs 40 MB: xarray doesn't compress it without an encoding (zlib). The
# Zarr weighs 21 MB, compressed by default (Zstandard), in 14 files: 6 chunks of 120
# months for anomaly, one per coordinate, and the metadata (zarr.json).
# Both read-backs are identical, attributes included.
# December 2015: a warm tongue, up to +3.6 °C, along the equator, from the coast of
# South America to the middle of the Pacific.

## 5. Under the NetCDF, HDF5

A NetCDF-4 file is an HDF5 file that follows conventions. Open `anomalies.nc` with `h5py`, without xarray:

- what objects does it hold?
- what shape, type and attributes does the `anomaly` dataset have?
- what are the first three values of `time` in the file, and its `units` attribute? What does xarray make of them?
- read the value of December 2015 (time index 551) on the equator, at longitude 250 (indices 44 and 125), and compare it with the one xarray gives, `nc_back.anomaly.sel(time="2015-12", lat=0, lon=250)`.

So what work does xarray do for you when reading?

**Useful documentation**

- [h5py quick start](https://docs.h5py.org/en/stable/quick.html)

In [ ]:
# Your code here

### Additional hints

`with h5py.File("anomalies.nc") as file:` opens the file for reading and closes it at the end of the block. An HDF5 file is browsed like a dictionary: `list(file)` gives its names, `file["anomaly"]` a dataset, which has `.shape`, `.dtype` and `.attrs` (a dictionary, whose `list` gives the names). Indexing a dataset, `variable[551, 44, 125]`, reads from disk only what you ask for. `DIMENSION_LIST` links each axis of `anomaly` to the dataset of its coordinate; the `_Netcdf4…` attributes are NetCDF's internal conventions.

```python
with h5py.File("anomalies.nc") as file:
  print(list(file))
  variable = file["anomaly"]
  print(variable.shape, variable.dtype, list(variable.attrs))
  print(file["time"][:3], file["time"].attrs[...])
  print(variable[..., ..., ...])
print(nc_back.time.values[:3])
print(nc_back.anomaly.sel(...).item())
```

### Solution

In [ ]:
with h5py.File("anomalies.nc") as file:
  print(list(file))
  variable = file["anomaly"]
  print(variable.shape, variable.dtype, list(variable.attrs))
  print(variable.attrs["long_name"], variable.attrs["units"])
  print(file["time"][:3], file["time"].attrs["units"])
  print(variable[551, 44, 125])
print(nc_back.time.values[:3])
print(nc_back.anomaly.sel(time="2015-12", lat=0, lon=250).item())
# Four datasets: anomaly and its three coordinates, lat, lon and time.
# anomaly: (624, 89, 180), float32; its attributes are ours (long_name, units),
# _FillValue (NaN, the missing value) and NetCDF's (DIMENSION_LIST…).
# In the file, time holds 62091, 62122, 62150: days since 1 January 1800, which xarray
# decodes into dates (1970-01-01, 1970-02-01, 1970-03-01).
# Both readings give the same value, +3.46 °C.
# xarray names the axes and attaches their coordinates to them, decodes the dates and
# the missing values, and lets you select by label rather than by index.

## 6. Optional: animate the anomalies of 2015 and 2016

Scroll, month by month, through the anomaly maps from January 2015 to December 2016, the strongest El Niño episode of the series. With `matplotlib.animation.FuncAnimation`, a function redraws the figure for each frame; `HTML(animation.to_jshtml())` shows the animation in the notebook, with its playback buttons.

Keep the same colour scale for every frame (`vmin`, `vmax`): why is it essential? Where does the warm anomaly appear, how does it spread, and what becomes of it in 2016?

**Useful documentation**

- [`matplotlib.animation.FuncAnimation`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html)
- [`Animation.to_jshtml`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.Animation.html#matplotlib.animation.Animation.to_jshtml)

In [ ]:
# Your code here

### Additional hints

First load the 24 maps into memory (`.load()`), so as not to read the Zarr again for each frame. Plot the first one with `.plot(ax=ax, cmap="RdBu_r", vmin=-4, vmax=4)`: it returns the drawn object (a `QuadMesh`), whose values the animation's function replaces, `image.set_array(maps.isel(time=i).values.ravel())`, before changing the title (`ax.set_title`). `FuncAnimation(fig, function, frames=24, interval=300)` calls the function for each frame, one every 300 ms. `plt.close(fig)` keeps the still figure from being shown on top of the animation.

```python
maps = zarr_back.anomaly.sel(time=slice(..., ...)).load()
fig, ax = plt.subplots(figsize=(8, 4))
image = maps.isel(time=0).plot(ax=ax, cmap=..., vmin=..., vmax=...)


def draw(i):
  image.set_array(...)
  ax.set_title(...)
  return [image]


animation = FuncAnimation(fig, draw, frames=..., interval=300)
plt.close(fig)
HTML(animation.to_jshtml())
```

### Solution

In [ ]:
maps = zarr_back.anomaly.sel(time=slice("2015-01", "2016-12")).load()
fig, ax = plt.subplots(figsize=(8, 4))
image = maps.isel(time=0).plot(ax=ax, cmap="RdBu_r", vmin=-4, vmax=4)


def draw(i):
  image.set_array(maps.isel(time=i).values.ravel())
  ax.set_title(str(maps.time.values[i])[:7])
  return [image]


animation = FuncAnimation(fig, draw, frames=maps.sizes["time"], interval=300)
plt.close(fig)
# Without a fixed scale, each frame would have its own: one colour wouldn't mean the
# same number of degrees from one frame to the next, and the anomaly's growth wouldn't
# show.
# The warm anomaly appears in spring 2015 near the coasts of Ecuador and Peru,
# stretches westward along the equator to the middle of the Pacific, peaks at the end
# of 2015, then fades in spring 2016; by the end of 2016, the eastern equatorial
# Pacific is colder than normal: a weak La Niña.
HTML(animation.to_jshtml())